# 00 Validate VDJdb AIRR vs Embeddings

Read the TRB VDJdb YLQ and GLC AIRR files, rebuild the cleaned repertoires through the same parser/filtering stage used before embedding, and check that the cleaned repertoire lengths match the embedding parquet files.

In [1]:
from pathlib import Path
import os
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "benchmark" / "data_sources.py").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing benchmark/data_sources.py")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Benchmark repo root: {REPO_ROOT}")


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from benchmark.data_sources import DEFAULT_VDJDB_EMBED_DIR, VDJDB_TARGETS, resolve_vdjdb_embedding_path
from benchmark.airr_utils import (
    best_clone_id_key,
    clone_id_candidates,
    read_airr_like_table,
    split_embedding_metadata,
    standardize_metadata_frame,
)

MIR_ENV_AVAILABLE = True
try:
    from mir.common.segments import SegmentLibrary
    from redcea.utils.tcremp import get_representations_df, load_analysis_repertoire
except ModuleNotFoundError:
    MIR_ENV_AVAILABLE = False
    print("mir/tcremp environment is not available locally; notebook will run in empty local mode.")


VDJDB_AIRR_DIR = Path("/projects/immunestatus/vdjdb/airr_format")
VDJDB_EMBED_DIR = Path(DEFAULT_VDJDB_EMBED_DIR)
OUTPUT_DIR = REPO_ROOT / "results" / "vdjdb_repertoire_alignment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHAIN = "TRB"
SPECIES = "HomoSapiens"
LOWER_LEN_CDR3 = 5
HIGHER_LEN_CDR3 = 30

TARGETS = {}
if MIR_ENV_AVAILABLE:
    for target_key, target in VDJDB_TARGETS.items():
        airr_path = VDJDB_AIRR_DIR / f"trb_vdjdb_{target['epitope_sequence']}.tsv"
        embedding_path = resolve_vdjdb_embedding_path(target_key, VDJDB_EMBED_DIR)["sample_embedding"]
        if airr_path.exists() and embedding_path.exists():
            TARGETS[target_key] = {
                "epitope": target["epitope_sequence"],
                "airr_path": airr_path,
                "embedding_path": embedding_path,
            }

if not TARGETS:
    print("VDJdb AIRR/embedding inputs are not available locally; notebook will run in empty local mode.")

pd.DataFrame(
    {
        "target_key": list(TARGETS),
        "airr_path": [str(TARGETS[key]["airr_path"]) for key in TARGETS],
        "embedding_path": [str(TARGETS[key]["embedding_path"]) for key in TARGETS],
    }
)



In [3]:
def load_cleaned_repertoire(airr_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    genes = CHAIN.split("_")
    locus = {"TRA": "alpha", "TRB": "beta", "TRA_TRB": None}[CHAIN]
    lib = SegmentLibrary.load_default(genes=genes, organisms=SPECIES)
    repertoire = load_analysis_repertoire(
        str(airr_path),
        lib,
        locus,
        mapping_column=None,
        llen=LOWER_LEN_CDR3,
        hlen=HIGHER_LEN_CDR3,
    )
    cleaned_rep = get_representations_df(repertoire, locus).reset_index(drop=True)
    cleaned_std = standardize_metadata_frame(cleaned_rep, chain_default=CHAIN).reset_index(drop=True)
    return cleaned_rep, cleaned_std


def maybe_standardize_embedding_metadata(frame: pd.DataFrame) -> pd.DataFrame | None:
    try:
        return standardize_metadata_frame(frame, chain_default=CHAIN).reset_index(drop=True)
    except Exception:
        return None


def build_comparison_table(
    cleaned_rep: pd.DataFrame,
    cleaned_std: pd.DataFrame,
    embedding_meta: pd.DataFrame,
    embedding_rows: int,
) -> tuple[pd.DataFrame, str, int]:
    embedding_candidates = clone_id_candidates(embedding_meta, sample_label="sample")
    cleaned_candidates = clone_id_candidates(cleaned_rep, sample_label="sample")
    match_columns, overlap = best_clone_id_key(embedding_candidates, cleaned_candidates)

    if match_columns is not None and overlap == embedding_rows:
        embedding_key, cleaned_key = match_columns
        cleaned_with_key = cleaned_std.copy()
        cleaned_with_key["cleaned_clone_id_match"] = cleaned_candidates[cleaned_key].astype(str)
        embedding_order = pd.DataFrame(
            {
                "embedding_row": np.arange(embedding_rows, dtype=np.int64),
                "embedding_clone_id_match": embedding_candidates[embedding_key].astype(str),
            }
        )
        comparison = embedding_order.merge(
            cleaned_with_key,
            left_on="embedding_clone_id_match",
            right_on="cleaned_clone_id_match",
            how="left",
            sort=False,
        )
        match_key = f"clone_id:{embedding_key}->{cleaned_key}"
    else:
        comparison = cleaned_std.reset_index(drop=True).copy()
        comparison.insert(0, "embedding_row", np.arange(embedding_rows, dtype=np.int64))
        match_key = "row_order"

    embedding_std = maybe_standardize_embedding_metadata(embedding_meta)
    if embedding_std is not None and len(embedding_std) == embedding_rows:
        embedding_std = embedding_std.add_prefix("embedding_")
        comparison = pd.concat([comparison.reset_index(drop=True), embedding_std.reset_index(drop=True)], axis=1)
        comparison["cdr3_match"] = comparison["cdr3"].astype(str).eq(comparison["embedding_cdr3"].astype(str))
        comparison["v_gene_match"] = comparison["v_gene"].astype(str).eq(comparison["embedding_v_gene"].astype(str))
        comparison["j_gene_match"] = comparison["j_gene"].astype(str).eq(comparison["embedding_j_gene"].astype(str))
    return comparison, match_key, overlap


def run_one_target(target_key: str, config: dict[str, object]) -> tuple[dict[str, object], pd.DataFrame | None]:
    airr_path = Path(config["airr_path"])
    embedding_path = Path(config["embedding_path"])
    raw_airr = read_airr_like_table(airr_path).reset_index(drop=True)
    cleaned_rep, cleaned_std = load_cleaned_repertoire(airr_path)
    embedding_frame = pd.read_parquet(embedding_path)
    embedding_meta, embedding_array = split_embedding_metadata(embedding_frame)

    cleaned_out = OUTPUT_DIR / f"{target_key.lower()}_cleaned_repertoire.tsv"
    cleaned_rep.to_csv(cleaned_out, sep="\t", index=False)

    length_match = len(cleaned_std) == len(embedding_frame)
    comparison = None
    match_key = "length_mismatch"
    clone_overlap = 0
    cdr3_mismatch_rows = np.nan

    if length_match:
        comparison, match_key, clone_overlap = build_comparison_table(
            cleaned_rep=cleaned_rep,
            cleaned_std=cleaned_std,
            embedding_meta=embedding_meta,
            embedding_rows=len(embedding_frame),
        )
        comparison_out = OUTPUT_DIR / f"{target_key.lower()}_embedding_vs_cleaned.tsv"
        comparison.to_csv(comparison_out, sep="\t", index=False)
        if "cdr3_match" in comparison.columns:
            cdr3_mismatch_rows = int((~comparison["cdr3_match"]).sum())

    summary = {
        "target_key": target_key,
        "epitope": config["epitope"],
        "raw_airr_rows": int(len(raw_airr)),
        "cleaned_rows": int(len(cleaned_std)),
        "embedding_rows": int(len(embedding_frame)),
        "embedding_dim": int(embedding_array.shape[1]),
        "rows_removed_by_cleaning": int(len(raw_airr) - len(cleaned_std)),
        "length_match": bool(length_match),
        "match_key": match_key,
        "clone_id_overlap": int(clone_overlap),
        "cdr3_mismatch_rows": cdr3_mismatch_rows,
        "airr_path": str(airr_path),
        "embedding_path": str(embedding_path),
        "cleaned_output": str(cleaned_out),
    }
    return summary, comparison


In [4]:
summaries = []
comparisons = {}

for target_key, config in TARGETS.items():
    summary, comparison = run_one_target(target_key, config)
    summaries.append(summary)
    comparisons[target_key] = comparison

if summaries:
    summary_df = pd.DataFrame(summaries).sort_values("target_key").reset_index(drop=True)
else:
    summary_df = pd.DataFrame(
        columns=[
            "target_key",
            "epitope",
            "raw_airr_rows",
            "cleaned_rows",
            "embedding_rows",
            "embedding_dim",
            "rows_removed_by_cleaning",
            "length_match",
            "match_key",
            "clone_id_overlap",
            "cdr3_mismatch_rows",
            "airr_path",
            "embedding_path",
            "cleaned_output",
        ]
    )
summary_df



In [5]:
for target_key in sorted(comparisons):
    print(f"=== {target_key} ===")
    comparison = comparisons[target_key]
    if comparison is None:
        print("length mismatch: comparison table was not built")
        print()
        continue
    display(comparison.head())
    if "cdr3_match" in comparison.columns:
        mismatch = comparison.loc[~comparison["cdr3_match"]].head()
        if len(mismatch):
            print("First CDR3 mismatches:")
            display(mismatch)
    print()
